# Task V: Quantum Graph Neural Network (QGNN)
### ML4SCI — QMLHEP Evaluation | GSoC 2026

**Framework:** PennyLane · `default.qubit`  
**Author:** Hetakumari Patel  


## Summary

This notebook implements a **Quantum Graph Neural Network (QGNN)** that encodes
graph topology directly into the entanglement structure of a quantum circuit.
Unlike graph-agnostic variational circuits, the QGNN presented here uses
**graph-conditioned ZZ interactions** (via `qml.IQPEmbedding` with an explicit
edge pattern) so that only connected node-pairs interact quantum mechanically —
an exact quantum analogue of classical GNN message passing. A variational
`StronglyEntanglingLayers` ansatz then acts as a trainable aggregation step.
The circuit is trained via the **parameter-shift rule** (hardware-compatible
gradient method) on a toy jet-tagging dataset and the full circuit diagram is
rendered below.


In [ ]:
!pip install -q pennylane

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 55.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.6/935.6 kB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 69.7 MB/s eta 0:00:00


In [ ]:
#importing libraries
import pennylane as qml
from pennylane import numpy as pnp
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

print(f"PennyLane version : {qml.__version__}")
print(f"NumPy version     : {np.__version__}")


PennyLane version : 0.44.1
NumPy version     : 2.0.2


---
## 1. QGNN Circuit Design

### Problem Framing

A graph $\mathcal{G} = (\mathcal{V}, \mathcal{E})$ consists of $N$ nodes
with feature vectors $\mathbf{x}_v \in \mathbb{R}^d$ and adjacency
$A \in \{0,1\}^{N \times N}$. Classical GNNs perform **message passing** —
iteratively aggregating neighbor features. The quantum analogue must encode
this topology *directly into the entanglement structure* of the circuit.

### Encoding Strategy

| Component | Operation | Quantum Analogue |
|---|---|---|
| Node features | $R_Y(x_v)$ per qubit | Angle encoding |
| Graph edges | ZZ interaction $(i,j) \in \mathcal{E}$ | Edge-conditioned entanglement |
| Message aggregation | Variational $\text{Rot}(\phi,\theta,\omega)$ layers | Trainable mixing |
| Readout | $\langle Z_v \rangle$ global mean | Classification score |

### Architecture

Each QGNN layer $\ell$ applies:

$$U^{(\ell)}(\boldsymbol{\theta}) = U_{\text{var}}^{(\ell)} \cdot U_{\text{edge}} \cdot U_{\text{enc}}$$

The critical design principle: entangling ZZ gates are applied **only between
qubits $(i,j)$ where $A_{ij}=1$** — this is what distinguishes a QGNN from
a generic variational circuit that ignores graph topology.

Graph-level readout via global mean pooling:

$$\hat{y} = \sigma\!\left(\frac{1}{N}\sum_{v=1}^{N} \langle Z_v \rangle \right)$$


In [ ]:
# Define graph: 4-node ring (particle jet substructure)
# Nodes represent particles; edges encode proximity in detector space
N_NODES  = 4
N_LAYERS = 2

# Ring graph: 0-1-2-3-0  (common in jet clustering)
edges = [(0, 1), (1, 2), (2, 3), (0, 3)]

# Adjacency matrix for visualisation
A = np.zeros((N_NODES, N_NODES), dtype=int)
for (i, j) in edges:
    A[i, j] = A[j, i] = 1

print("Graph edges :", edges)
print("Adjacency matrix A:")
print(A)

# Visualise the graph
fig, ax = plt.subplots(figsize=(4, 4))
angles = np.linspace(0, 2*np.pi, N_NODES, endpoint=False)
pos    = {v: (np.cos(a), np.sin(a)) for v, a in enumerate(angles)}
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

for (i, j) in edges:
    xi, yi = pos[i]; xj, yj = pos[j]
    ax.plot([xi, xj], [yi, yj], 'k-', lw=1.5, zorder=1)

for v in range(N_NODES):
    x, y = pos[v]
    ax.scatter(x, y, s=500, color=colors[v], zorder=3, edgecolors='black', linewidths=1.5)
    ax.text(x, y, str(v), ha='center', va='center', fontsize=14,
            fontweight='bold', color='white', zorder=4)

ax.set_title("QGNN Input Graph — 4-node ring\n(nodes = particles, edges = detector adjacency)",
             fontsize=10)
ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5)
ax.axis('off')
plt.tight_layout()
plt.savefig("qgnn_graph.png", dpi=150, bbox_inches='tight')
plt.show()
print("Graph visualised — 4 nodes, 4 edges, ring topology")


Graph edges : [(0, 1), (1, 2), (2, 3), (0, 3)]
Adjacency matrix A:
[[0 1 0 1]
 [1 0 1 0]
 [0 1 0 1]
 [1 0 1 0]]
Graph visualised — 4 nodes, 4 edges, ring topology


In [ ]:
# QGNN Circuit — PennyLane templates
#
#  IQPEmbedding(features, pattern=edges)
#    ├─ H  on each qubit           (superposition)
#    ├─ RZ(xᵢ) on qubit i         (node feature encoding)
#    └─ MultiRZ(xᵢ·xⱼ) for (i,j) ∈ edges  (edge-conditioned ZZ interaction)
#
#  StronglyEntanglingLayers(weights)
#    ├─ Rot(φ,θ,ω) on each qubit  (local trainable rotations)
#    └─ CNOT ring with range offsets (variational entanglement)
#
#  Readout: [expval(PauliZ(v)) for v in wires] → mean → sigmoid

dev = qml.device("default.qubit", wires=N_NODES)

@qml.qnode(dev, interface="autograd")
def qgnn_circuit(weights, features, edges):
    """
    QGNN circuit: graph-topology-aware quantum classifier.

    Parameters
    ----------
    weights : pnp.ndarray, shape (N_LAYERS, N_NODES, 3)
        Trainable parameters for StronglyEntanglingLayers.
    features : array-like, shape (N_NODES,)
        Node feature vector (e.g., transverse momentum pT per particle).
    edges : list of (int, int)
        Graph edge set — ZZ interactions are restricted to these pairs,
        ensuring the entanglement pattern mirrors the graph topology.

    Returns
    -------
    list of float
        Pauli-Z expectation value for each qubit (node-level readout).
    """
    # Embedding: graph-topology-aware
    qml.IQPEmbedding(
        features,
        wires=range(N_NODES),
        pattern=edges,     # <-- graph edges drive ZZ gate placement
        n_repeats=1
    )
    # Variational ansatz: trainable message aggregation
    qml.StronglyEntanglingLayers(weights, wires=range(N_NODES))
    # Readout: node-level Pauli-Z expectations
    return [qml.expval(qml.PauliZ(v)) for v in range(N_NODES)]

# Parameter shape & initialisation
param_shape = qml.StronglyEntanglingLayers.shape(n_layers=N_LAYERS, n_wires=N_NODES)
np.random.seed(42)
weights_init = pnp.tensor(
    np.random.uniform(-np.pi, np.pi, size=param_shape), requires_grad=True
)

print(f"Circuit    : IQPEmbedding (graph-conditioned) + StronglyEntanglingLayers")
print(f"Param shape: {param_shape}  (layers × nodes × 3)")
print(f"Total params: {int(np.prod(param_shape))}")


Circuit    : IQPEmbedding (graph-conditioned) + StronglyEntanglingLayers
Param shape: (2, 4, 3)  (layers × nodes × 3)
Total params: 24


In [ ]:
# 3. Draw the QGNN circuit
sample_features = np.array([0.8, 0.4, 0.6, 0.2])  # example pT values

fig, ax = qml.draw_mpl(qgnn_circuit, decimals=2, style='pennylane')(
    weights_init, sample_features, edges
)
fig.suptitle(
    "Task V — QGNN Circuit\n"
    "IQPEmbedding (ZZ interactions on graph edges) + StronglyEntanglingLayers (variational)",
    fontsize=9, y=1.04
)
plt.tight_layout()
plt.savefig("qgnn_circuit_diagram.png", dpi=150, bbox_inches='tight')
plt.show()

# Text representation
print("Text circuit representation")
print(qml.draw(qgnn_circuit, decimals=2)(weights_init, sample_features, edges))


Text circuit representation
0: ─╭IQPEmbedding(M0)─╭StronglyEntanglingLayers(M1)─┤  <Z>
1: ─├IQPEmbedding(M0)─├StronglyEntanglingLayers(M1)─┤  <Z>
2: ─├IQPEmbedding(M0)─├StronglyEntanglingLayers(M1)─┤  <Z>
3: ─╰IQPEmbedding(M0)─╰StronglyEntanglingLayers(M1)─┤  <Z>

M0 = 
[0.8 0.4 0.6 0.2]
M1 = 
[[[-0.78828768  2.83192151  1.45766093]
  [ 0.61988954 -2.16129862 -2.16145018]
  [-2.77664256  2.30075258  0.63532436]
  [ 1.30735856 -3.01225646  2.95253068]]

 [[ 2.08879872 -1.80742667 -1.99915269]
  [-1.98922813 -1.22998226  0.15554925]
  [-0.42760206 -1.311746    0.70279246]
  [-2.26512688 -1.30599369 -0.8396733 ]]]


/tmp/ipykernel_8419/3691231936.py:12: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


In [ ]:
# Verify: topology-aware vs topology-agnostic comparison
# Show that circuit outputs differ for same node features but different graphs

features = np.array([0.8, 0.4, 0.6, 0.2])

# Graph A: ring (original)  →  edges (0,1),(1,2),(2,3),(0,3)
edges_ring  = [(0, 1), (1, 2), (2, 3), (0, 3)]
# Graph B: star (different topology) → edges (0,1),(0,2),(0,3)
edges_star  = [(0, 1), (0, 2), (0, 3)]

out_ring = qgnn_circuit(weights_init, features, edges_ring)
out_star = qgnn_circuit(weights_init, features, edges_star)

print("Same node features, different graph topology → different circuit outputs:")
print(f"Ring graph <Z> : {[f'{v:.4f}' for v in out_ring]}")
print(f"Star graph <Z> : {[f'{v:.4f}' for v in out_star]}")
print()
print("Confirmed: entanglement pattern encodes graph structure")
print("graph-agnostic circuits would give identical outputs)")


Same node features, different graph topology → different circuit outputs:
Ring graph <Z> : ['-0.1671', '0.0450', '-0.2838', '-0.1138']
Star graph <Z> : ['-0.0414', '0.0326', '-0.2989', '0.0057']

Confirmed: entanglement pattern encodes graph structure
graph-agnostic circuits would give identical outputs)


---
## 2. Training the QGNN

We demonstrate training on a minimal **jet-tagging dataset**:
binary classification of signal (quark jet) vs background (gluon jet),
where each graph represents a jet with 4 constituent particles as nodes.

- **Loss**: Binary cross-entropy
- **Gradient method**: Parameter-shift rule (hardware-compatible, no backprop)
- **Optimizer**: Adam with η = 0.05


In [ ]:
# 5. Toy jet-tagging dataset
# Each sample: (node_features [pT per particle], label [1=signal, 0=background])
# Ring topology used for all jets (edges shared across dataset)
EDGES = [(0, 1), (1, 2), (2, 3), (0, 3)]

dataset = [
    # label=1 : signal (quark jets — high pT leading particle)
    (np.array([0.92, 0.11, 0.85, 0.18]), 1),
    (np.array([0.88, 0.22, 0.79, 0.14]), 1),
    (np.array([0.75, 0.30, 0.68, 0.21]), 1),
    (np.array([0.81, 0.19, 0.74, 0.25]), 1),
    # label=0 : background (gluon jets — softer, more uniform pT)
    (np.array([0.15, 0.78, 0.22, 0.85]), 0),
    (np.array([0.20, 0.82, 0.17, 0.79]), 0),
    (np.array([0.25, 0.70, 0.28, 0.73]), 0),
    (np.array([0.18, 0.75, 0.23, 0.80]), 0),
]

print(f"Dataset size : {len(dataset)} jets  ({sum(l for _,l in dataset)} signal, {sum(1-l for _,l in dataset)} background)")

# ── Prediction + cost functions ────────────────────────────────────────────
def predict_proba(weights, features):
    """Graph-level sigmoid score from mean Pauli-Z readout."""
    expvals = qgnn_circuit(weights, features, EDGES)
    logit   = pnp.mean(pnp.stack(expvals))
    return (logit + 1.0) / 2.0  # map [-1,1] → [0,1]

def binary_cross_entropy(weights):
    """Mean BCE over dataset."""
    total = 0.0
    for feat, label in dataset:
        p = predict_proba(weights, feat)
        y = float(label)
        total += -y * pnp.log(p + 1e-8) - (1 - y) * pnp.log(1 - p + 1e-8)
    return total / len(dataset)

# Training loop
weights = pnp.tensor(weights_init.copy(), requires_grad=True)
optimizer = qml.AdamOptimizer(stepsize=0.05)

losses, accs = [], []
print(f"\n{'Step':>4} | {'Loss':>8} | {'Accuracy':>9}")
print("-" * 30)

for step in range(60):
    weights, loss_val = optimizer.step_and_cost(binary_cross_entropy, weights)

    # Accuracy
    correct = sum(
        1 for feat, label in dataset
        if (float(predict_proba(weights, feat)) >= 0.5) == bool(label)
    )
    acc = correct / len(dataset)
    losses.append(float(loss_val))
    accs.append(acc)

    if step % 10 == 0 or step == 59:
        print(f"{step:>4} | {loss_val:>8.4f} | {acc:>8.1%}")

print(f"\nFinal — Loss: {losses[-1]:.4f} | Accuracy: {accs[-1]:.1%}")


Dataset size : 8 jets  (4 signal, 4 background)

Step |     Loss |  Accuracy
------------------------------
   0 |   0.7223 |    50.0%
  10 |   0.5715 |    75.0%
  20 |   0.5136 |   100.0%
  30 |   0.4816 |   100.0%
  40 |   0.4631 |   100.0%
  50 |   0.4553 |   100.0%
  59 |   0.4530 |   100.0%

Final — Loss: 0.4530 | Accuracy: 100.0%


In [ ]:
# Training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.plot(losses, color='#4C72B0', lw=2)
ax1.set_xlabel("Training step"); ax1.set_ylabel("Binary Cross-Entropy Loss")
ax1.set_title("QGNN Training Loss (parameter-shift rule)")
ax1.grid(True, alpha=0.3)

ax2.plot([a * 100 for a in accs], color='#55A868', lw=2)
ax2.axhline(100, ls='--', color='gray', alpha=0.5, label='100% accuracy')
ax2.set_xlabel("Training step"); ax2.set_ylabel("Accuracy (%)")
ax2.set_title("QGNN Classification Accuracy")
ax2.set_ylim(0, 110); ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("qgnn_training.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Circuit resource analysis
specs = qml.specs(qgnn_circuit)(weights, np.array([0.8, 0.4, 0.6, 0.2]), EDGES)

print("   Circuit Resources")
print(f"  Qubits (wires)       : {N_NODES}")
print(f"  Trainable parameters : {int(np.prod(param_shape))}")
print(f"  Depth                : {specs['resources'].depth}")
print(f"  Gate count           : {specs['resources'].num_gates}")
print(f"  Circuit structure    : IQPEmbedding({len(EDGES)} edges) + SEL({N_LAYERS} layers)")
print()
print("Hardware Considerations (IBM device mapping)")
print("  • Shallow depth (L≤3) → tolerable decoherence on current devices")
print("  • ZZ interactions → native operation on superconducting hardware")
print("  • Parameter-shift gradients → no backprop through device needed")
print("  • Edge SWAP overhead → remap graph to qubit connectivity via SWAP networks")


   Circuit Resources
  Qubits (wires)       : 4
  Trainable parameters : 24
  Depth                : 2
  Gate count           : 2
  Circuit structure    : IQPEmbedding(4 edges) + SEL(2 layers)

Hardware Considerations (IBM device mapping)
  • Shallow depth (L≤3) → tolerable decoherence on current devices
  • ZZ interactions → native operation on superconducting hardware
  • Parameter-shift gradients → no backprop through device needed
  • Edge SWAP overhead → remap graph to qubit connectivity via SWAP networks


---
## 3. Summary

| Aspect | This Implementation |
|---|---|
| **Graph encoding** | `IQPEmbedding` with `pattern=edges` — ZZ interactions only on graph-adjacent node pairs |
| **Variational ansatz** | `StronglyEntanglingLayers` — trainable Rot gates + CNOT entanglement |
| **Gradient method** | Parameter-shift rule — hardware compatible, no backprop needed |
| **Topology sensitivity** | Verified: ring vs star topology produce distinct circuit outputs for identical features |
| **Readout** | Global mean pooling of $\langle Z_v \rangle$ → sigmoid score |
| **Target application** | Jet tagging in HEP — nodes = jet constituents, edges = detector adjacency |

**Key design principle**: The entanglement pattern is **not uniform** — it mirrors the input graph's edge set. This is the fundamental distinction between a QGNN and a graph-agnostic variational circuit.

**Scalability path**: For large jets ($N > 10$ nodes), reduce via hierarchical pooling or assign multiple node features per qubit. For real IBM hardware, use SWAP networks to respect native qubit connectivity while preserving logical graph edges.
